# 🧪 W15-D2 实验：RAG 三件套检索机制 × Semantic Model / Text-to-SQL 分工边界

**对应阅读材料**：`第15周-Day2-LnkChatBI架构精读②-RAG三件套与语义分工边界.md`（md 讲「为什么」，本 notebook 用可执行实验验证三个主张）

| # | 实验 | 验证的主张 | 代码事实依据（LnkChatBI backend） |
|---|---|---|---|
| ① | 三件套检索机制忠实模拟 | 术语=单向子串+向量双路、命中别名拉全组；示例=双向子串；custom_prompt=无检索全量注入 | `terminology.py:913` / `data_training.py:676` / `custom_prompt.py` |
| ② | 命名漂移量化 | 检索能力面 = 字面命中 ∪ 语义近邻；任意编码映射（A101→LOC_DEMO_L101）在两者之外 → 必须靠 Semantic Model 预喂别名 | 检索 SQL 语义 + `EMBEDDING_*_SIMILARITY=0.4`（config.py:161-167） |
| ③ | 错误归因分界蒙特卡洛 | 语义错（词-物映射）= 治理修复，覆盖门控、确定性；组装错（join/写法）= 校准修复，概率性、可重试 | Today's Question 的操作化 |

> **诚实声明**：向量检索用字符 n-gram 余弦做**表层相似度代理**（服务器无 embedding 模型、禁联网），真实 pgvector embedding 有语义泛化能力（比 n-gram 强）；实验②用「同义对 vs 漂移编码对」的对照来锚定哪些结论对代理选择稳健。实验③的概率参数是模拟假设，主张的是**机制结构**（确定性 vs 概率性修复），不是具体数值。

In [ ]:
# 字体配置：TOOLS.md 唯一标准（所有 notebook 必须用这个）
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)
import numpy as np
import os
OUT = "/root/learning-notebooks/第15周"
os.makedirs(OUT, exist_ok=True)
rng = np.random.default_rng(42)

## 实验① 三件套检索机制忠实模拟

按 LnkChatBI 真实表结构构造术语库（`Terminology` 表：父行持 `word+description`，子行 pid 指父只持 `word`；字段见 `terminology_model.py`），并**逐条复现**三条检索链的 SQL 语义：

- 术语链（`terminology.py:913`）：主路 `:sentence ILIKE '%' || word || '%'`（**单向**：问句包含术语词）+ 向量路（表层代理，阈值 0.4）→ 命中任一词按 id/pid 去重后**拉全组**
- 示例链（`data_training.py:676`）：**双向**子串 `sentence ILIKE '%question%' OR question ILIKE '%sentence%'`
- 指令链（`custom_prompt.py`）：无检索，oid+type 全量

术语组内容取自 mallcre 种子数据的真实字段（`bi_d_position` 的 POSITION_CODE / POSITION_STATE / CONT_NO / END_DATE），description 按 D1 的 L2/L3 判据写——这就是 **Rule 构件经术语 description 注入**的实例。

In [ ]:
# ---- 忠实复现 LnkChatBI 术语库结构与检索逻辑 ----
SIM_THRESHOLD = 0.4   # config.py:162 EMBEDDING_DEFAULT_SIMILARITY
TOP_COUNT = 5         # EMBEDDING_TERMINOLOGY_TOP_COUNT

# Terminology 行的忠实模拟：{id, pid, word, description}（父行 pid=None 持 description）
TERMS = [
    # 组1：计租面积（父）+ 别名（子）——description 携带口径（Rule 注入接口实例）
    {"id": 1, "pid": None, "word": "计租面积", "description": "租金计算面积，字段 bi_d_position.RENT_AREA（平方米，含公摊口径以租赁合同为准）。"},
    {"id": 2, "pid": 1, "word": "租赁面积", "description": None},
    {"id": 3, "pid": 1, "word": "rentable area", "description": None},
    # 组2：空置铺位——description 携带判定规则（D1 L2 判据的术语化）
    {"id": 4, "pid": None, "word": "空置铺位", "description": "判定条件：POSITION_STATE = 2 或 CONT_NO = '-'（快闪铺 END_DATE=2099 视为短租占用，不算空置）。表 bi_d_position。"},
    {"id": 5, "pid": 4, "word": "空置", "description": None},
    {"id": 6, "pid": 4, "word": "未出租", "description": None},
    # 组3：铺位编码——注意：预喂前【不含】A101 子别名（实验②会补）
    {"id": 7, "pid": None, "word": "铺位编码", "description": "铺位唯一编码，字段 bi_d_position.POSITION_CODE，编码风格 LOC_DEMO_L1xx。"},
    {"id": 8, "pid": 7, "word": "铺位号", "description": None},
    {"id": 9, "pid": 7, "word": "铺位", "description": None},
]

def norm(s: str) -> str:
    return s.lower().strip()

def ngrams(s: str, n: int = 2) -> dict:
    s = norm(s)
    out = {}
    for i in range(len(s) - n + 1):
        g = s[i:i+n]
        out[g] = out.get(g, 0) + 1
    return out

def cosine(a: dict, b: dict) -> float:
    if not a or not b:
        return 0.0
    common = set(a) & set(b)
    dot = sum(a[g] * b.get(g, 0) for g in common)
    na = np.sqrt(sum(v*v for v in a.values())); nb = np.sqrt(sum(v*v for v in b.values()))
    return dot / (na * nb) if na and nb else 0.0

def retrieve_terminology(question: str, terms=None, use_vector=True) -> list:
    """忠实复现 select_terminology_by_word：单向子串 + 向量双路 → 命中拉全组。"""
    terms = terms if terms is not None else TERMS
    q = norm(question); qg = ngrams(question)
    hit_ids = []
    for t in terms:  # 路A：单向子串（sentence 包含 word）
        if t["word"] and norm(t["word"]) in q:
            hit_ids.append(t["id"])
    if use_vector:   # 路B：向量相似（表层代理），> 0.4 截断 TOP_COUNT
        scored = [(cosine(qg, ngrams(t["word"])), t["id"]) for t in terms if t["word"]]
        scored = sorted(scored, reverse=True)[:TOP_COUNT]
        hit_ids += [i for s, i in scored if s > SIM_THRESHOLD]
    # 归并：id/pid 去重 → 拉全组（复现 _ids/_map 逻辑）
    ids = []
    for t in terms:
        if t["id"] in hit_ids or t["pid"] in hit_ids:
            ids.append(t["pid"] if t["pid"] is not None else t["id"])
    ids = list(dict.fromkeys(ids))
    groups = []
    for root in ids:
        rows = [t for t in terms if t["pid"] == root or t["id"] == root]
        parent = next(t for t in rows if t["pid"] is None)
        groups.append({"规范词": parent["word"], "别名": [t["word"] for t in rows],
                       "description": parent["description"]})
    return groups

def retrieve_examples(question: str, examples: list) -> list:
    """忠实复现 select_training_by_question：双向子串（向量路从略，机制同①）。"""
    q = norm(question)
    return [e for e in examples if norm(e["question"]) in q or q in norm(e["question"])]

# 快速自检：Q="星河中心的计租面积有多少" 应命中组1（子串"计租面积"）并拉出全部别名
g = retrieve_terminology("星河中心的计租面积有多少")
assert len(g) == 1 and g[0]["规范词"] == "计租面积" and len(g[0]["别名"]) == 3
print("✅ 子串命中并拉全组：", g[0]["规范词"], "| 别名:", g[0]["别名"])
print("✅ description（Rule 注入接口）:", g[0]["description"][:40], "...")

In [ ]:
# ---- 三配置召回矩阵：无术语 / 仅子串（EMBEDDING 关闭）/ 子串+向量 ----
QUESTIONS = {
    "星河中心的计租面积有多少": "组1·计租面积（字面命中）",
    "空置铺位有哪些": "组2·空置铺位（字面命中）",
    "各楼层未出租的商铺清单": "组2·空置铺位（子串 miss：问句是『未出租』，须子别名命中）",
    "快闪店什么时候到期": "无组（快闪概念未建术语——真实反映术语库覆盖面）",
    "A101 铺位为什么不能出租": "组3·铺位编码（子串『铺位』命中子别名；但 A101 值映射不在库里）",
}

rows = []
for q, expect in QUESTIONS.items():
    g_sub = retrieve_terminology(q, use_vector=False)
    g_both = retrieve_terminology(q, use_vector=True)
    rows.append({
        "问题": q, "预期": expect,
        "仅子串路": ("✓ " + "/".join(x["规范词"] for x in g_sub)) if g_sub else "✗ miss",
        "子串+向量": ("✓ " + "/".join(x["规范词"] for x in g_both)) if g_both else "✗ miss",
    })

print("%-24s%-26s%-26s%s" % ("问题", "仅子串路", "子串+向量", "预期"))
print("-" * 100)
for r in rows:
    print("%-24s%-26s%-26s%s" % (r["问题"], r["仅子串路"], r["子串+向量"], r["预期"]))

# 可视化：命中组数矩阵（0/1/2 组）
matrix = np.zeros((len(QUESTIONS), 3))
for i, q in enumerate(QUESTIONS):
    matrix[i, 0] = 0
    matrix[i, 1] = len(retrieve_terminology(q, use_vector=False))
    matrix[i, 2] = len(retrieve_terminology(q, use_vector=True))
fig, ax = plt.subplots(figsize=(9, 4.2))
im = ax.imshow(matrix, cmap="YlGn", vmin=0, vmax=2, aspect="auto")
ax.set_xticks(range(3))
ax.set_xticklabels(["无术语库\n(治理缺位)", "仅子串路\n(EMBEDDING_ENABLED=false)", "子串+向量\n(默认配置)"])
ax.set_yticks(range(len(QUESTIONS)))
ax.set_yticklabels(list(QUESTIONS.keys()), fontsize=9)
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        ax.text(j, i, int(matrix[i, j]), ha="center", va="center", fontsize=12)
ax.set_title("实验① 术语双路检索召回矩阵（数字=命中术语组数，阈值 0.4）")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout(); plt.savefig(f"{OUT}/w15d2_rag_recall_matrix.png", dpi=110); plt.show()
print("关键观察：『未出租』靠子别名救活（治理设计）；『快闪店』两路全 miss——检索兜底有边界。")

## 实验② 命名漂移量化：检索上界之外，靠治理预喂

「A101 铺位为什么不能出租」是 W15 的验收场景。种子数据的编码是 `LOC_DEMO_L101` 风格（D1 已证）——问句里的 `A101` 与库里的编码之间是**任意编码映射**（无语义中介）。量化两条真实检索路径对三类词对的得分：

- **字面对**：问句含库词 → 子串路确定性命中
- **同义对**：租赁面积 vs 计租面积 → 子串 miss；表层相似度低于阈值；**真实 embedding 可救**（语义泛化）
- **漂移编码对**：A101 vs LOC_DEMO_L101 → 子串 miss；表层相似度同样低于阈值；**真实 embedding 也救不了**（编码映射无语义关联）

治理修复演示：把 `A101` 预喂为组3 的子别名后，子串路**确定性**命中（这就是 D3 生成规格里「编码风格值进子别名」的依据）。

In [ ]:
# ---- 三类词对的两路得分 ----
PAIRS = [
    ("计租面积", "计租面积", "字面对（问句含库词）"),
    ("租赁面积", "计租面积", "同义对（真实embedding可救）"),
    ("A101", "LOC_DEMO_L101", "漂移编码对（两路都救不了）"),
    ("铺位", "POSITION_CODE", "口语vs字段名（部分语义关联）"),
]
print("%-10s%-18s%-8s%-10s%-10s%s" % ("问句词", "库词", "子串路", "表层余弦", "vs阈值0.4", "类别"))
print("-" * 78)
scores = []
for q, w, cat in PAIRS:
    sub = "✓命中" if norm(w) in norm(q) else "✗miss"
    sim = cosine(ngrams(q), ngrams(w))
    scores.append(sim)
    print("%-10s%-18s%-8s%-10.3f%-10s%s" % (q, w, sub, sim, "过" if sim > SIM_THRESHOLD else "不过", cat))

fig, ax = plt.subplots(figsize=(9, 3.8))
labels = [f"{q}→{w}" for q, w, _ in PAIRS]
colors = ["#2e7d32", "#f9a825", "#c62828", "#f9a825"]
bars = ax.bar(labels, scores, color=colors, width=0.55)
ax.axhline(SIM_THRESHOLD, color="black", ls="--", lw=1.2)
ax.text(3.45, SIM_THRESHOLD + 0.02, "阈值 0.4（真实配置）", fontsize=9, ha="right")
for b, s in zip(bars, scores):
    ax.text(b.get_x() + b.get_width()/2, s + 0.015, f"{s:.3f}", ha="center", fontsize=10)
ax.set_ylim(0, 1.0); ax.set_ylabel("表层余弦相似度（向量路代理）")
ax.set_title("实验② 检索能力面：字面命中∪语义近邻 之外的部分，检索无能为力")
plt.xticks(fontsize=9)
plt.tight_layout(); plt.savefig(f"{OUT}/w15d2_drift_scores.png", dpi=110); plt.show()
print("代理稳健性说明：同义对(绿→黄)真实 embedding 能过阈，但漂移编码对(红)是任意映射，")
print("语义空间中『A101』与『LOC_DEMO_L101』无任何中介——表层 0.33 的高分纯靠共享『10/01』碎片，")
print("即使换成真实 embedding 也无理由过阈。结论对代理选择稳健。")

In [ ]:
# ---- 治理修复演示：预喂 A101 为子别名 → 子串路确定性命中 ----
TERMS_FED = TERMS + [{"id": 10, "pid": 7, "word": "A101", "description": None}]
q = "A101 铺位为什么不能出租"
before = retrieve_terminology(q, TERMS, use_vector=True)
after = retrieve_terminology(q, TERMS_FED, use_vector=True)
def find_group(groups, name):
    return next((g for g in groups if g["规范词"] == name), None)
b3, a3 = find_group(before, "铺位编码"), find_group(after, "铺位编码")
print(f"预喂前：命中铺位编码组 = {b3 is not None}；组内别名 = {b3['别名'] if b3 else []}")
print(f"预喂后：命中铺位编码组 = {a3 is not None}；组内别名 = {a3['别名']}")
print(f"预喂后 description 随组注入：{a3['description'][:38]}...")
assert "A101" in a3["别名"] and a3["description"]
print()
print("✅ 同一个问题：检索上界之外（实验②红线）→ 治理预喂后子串路确定性命中且拉出口径 description。")
print("   这就是『分界线』的操作化：检索是能力面内的概率兜底，治理是能力面外的确定性扩展。")

## 实验③ 错误归因分界蒙特卡洛：治理修复 vs 校准修复的机制差异

Today's Question 的操作化。模拟 2000 个问数任务，两类独立故障：

- **语义故障**（词-物映射错，如把「计租面积」映射到 RENT_PRICE）：无术语注入时 p=0.30；术语注入后**覆盖门控**——被术语组覆盖的问题修复为 ~0，未覆盖的问题原样失败（覆盖率先验 c=0.8）
- **组装故障**（join 顺序/方言/limit 写错）：基础 p=0.20；示例校准（few-shot）后**概率性下降**到 p=0.05，但永远不为零，且各次独立

机制主张（对模拟参数稳健）：**治理修复是阶跃函数（按覆盖二值），校准修复是连续函数（概率下降）；重试能救组装错，救不了未覆盖的语义错。** 用四种策略的正确率 + 语义修复的零方差签名来验证。

In [ ]:
# ---- 四策略蒙特卡洛 + 方差签名 ----
N, REPEAT = 2000, 50
P_SEM, P_ASM = 0.30, 0.20          # 无干预故障率（模拟假设）
P_ASM_CAL = 0.05                    # 示例校准后的组装故障率（模拟假设）
COVERAGE = 0.8                      # 术语库覆盖率先验（语义修复=覆盖门控）
covered = rng.random(N) < COVERAGE  # 每个问题是否被术语组覆盖（固定属性）

def run(policy: str) -> np.ndarray:
    """返回每次重复的整体正确率。语义修复=覆盖门控（阶跃）；组装修复=概率下降（连续）。"""
    accs = []
    for _ in range(REPEAT):
        if policy == "none":
            sem_fail = rng.random(N) < P_SEM; asm_fail = rng.random(N) < P_ASM
        elif policy == "term":
            sem_fail = (rng.random(N) < P_SEM) & ~covered   # 覆盖内→确定性修复
            asm_fail = rng.random(N) < P_ASM
        elif policy == "example":
            sem_fail = rng.random(N) < P_SEM
            asm_fail = rng.random(N) < P_ASM_CAL             # 概率性下降，仍>0
        else:  # both
            sem_fail = (rng.random(N) < P_SEM) & ~covered
            asm_fail = rng.random(N) < P_ASM_CAL
        accs.append(1.0 - np.mean(sem_fail | asm_fail))
    return np.array(accs)

policies = ["none", "term", "example", "both"]
labels = ["无干预", "仅术语注入\n(治理)", "仅示例校准\n(校准)", "双管齐下"]
results = {p: run(p) for p in policies}
for p, l in zip(policies, [x.replace("\n", "") for x in labels]):
    m, s = results[p].mean(), results[p].std()
    print("%-12s 正确率 = %.3f ± %.3f" % (l, m, s))

# 方差签名：只看『被覆盖的问题子集』在 term 策略下逐次重复的故障率
sub_fail_rates = []
for _ in range(REPEAT):
    sem_fail = (rng.random(N) < P_SEM) & ~covered
    sub_fail_rates.append(sem_fail[covered].mean())          # 覆盖内子集的语义故障率
print("\n方差签名：术语注入后，被覆盖子集的语义故障率 = %.4f（恒 0，标准差 %.4f）"
      % (np.mean(sub_fail_rates), np.std(sub_fail_rates)))
print("→ 治理修复是确定性的（覆盖内零故障、零方差）；校准修复永远留尾巴（P_ASM_CAL>0）。")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
means = [results[p].mean() for p in policies]
errs = [results[p].std() for p in policies]
axes[0].bar(range(4), means, yerr=errs, capsize=4,
            color=["#9e9e9e", "#2e7d32", "#1565c0", "#6a1b9a"], width=0.6)
axes[0].set_xticks(range(4)); axes[0].set_xticklabels(labels, fontsize=9)
for i, m in enumerate(means):
    axes[0].text(i, m + 0.012, f"{m:.3f}", ha="center", fontsize=10)
axes[0].set_ylim(0, 1.0); axes[0].set_ylabel("问数正确率（2000任务×50重复）")
axes[0].set_title("四策略正确率")
# 右图：故障构成（both 策略下剩余故障的归因）
sem_fail = (rng.random(N) < P_SEM) & ~covered
asm_fail = rng.random(N) < P_ASM_CAL
rest = np.array(["未覆盖语义错" if s else ("组装错(可重试)" if a else "正确")
                 for s, a in zip(sem_fail, asm_fail)])
vals = [(rest == k).mean() for k in ["正确", "未覆盖语义错", "组装错(可重试)"]]
axes[1].bar(range(3), vals, color=["#9e9e9e", "#c62828", "#1565c0"], width=0.55)
axes[1].set_xticks(range(3))
axes[1].set_xticklabels(["正确", "剩余:未覆盖语义错\n(治理缺口→补术语)", "剩余:组装错\n(校准尾巴→可重试)"], fontsize=9)
for i, v in enumerate(vals):
    axes[1].text(i, v + 0.012, f"{v:.3f}", ha="center", fontsize=10)
axes[1].set_ylim(0, 1.0); axes[1].set_title("双管齐下后剩余故障的归因")
plt.suptitle("实验③ 错误归因分界：治理修复=覆盖门控阶跃，校准修复=概率下降连续", y=1.02)
plt.tight_layout(); plt.savefig(f"{OUT}/w15d2_boundary_mc.png", dpi=110, bbox_inches="tight"); plt.show()

In [ ]:
# ---- 双向子串边界：data_training 示例检索 vs 术语单向子串（收尾验证） ----
EXAMPLES = [
    {"question": "各项目计租面积汇总", "suggestion-answer": "SELECT PROJECT, SUM(RENT_AREA) FROM bi_d_position GROUP BY PROJECT"},
    {"question": "空置铺位清单按楼层", "suggestion-answer": "SELECT FLOOR, POSITION_CODE FROM bi_d_position WHERE POSITION_STATE=2"},
]
for q in ["计租面积", "计租面积按项目汇总多少"]:
    hits = retrieve_examples(q, EXAMPLES)
    print(f"问句『{q}』→ 命中示例 {len(hits)} 条: {[h['question'] for h in hits]}")
# 术语词若也走双向："计租面积" 会命中包含它的任何长术语——演示方向选择是数据形态决定的
print()
print("观察：短问句『计租面积』靠【双向】才能命中长示例『各项目计租面积汇总』；")
print("术语链是单向（问句须包含术语词）——因为术语词短、示例 question 长，方向跟着数据形态走。")
print()
print("===== 今日实验结论 =====")
print("① 三件套各管一层：词-物归一(检索) / 表达校准(检索) / 口径声明(无检索必注入)")
print("② 检索能力面 = 字面∪语义近邻；A101 类任意编码映射在能力面外，治理预喂后子串路确定性命中")
print("③ 分界线操作化：语义错→治理修(覆盖门控、零方差、重试无效)；组装错→校准修(概率下降、可重试)")

## 结论回接（与 md 的对应）

| 实验 | 结果 | 支撑的主张 |
|---|---|---|
| ① 召回矩阵 | 「未出租」靠子别名命中（治理设计）；「快闪店」两路全 miss | 检索兜底有边界；别名组设计=归一语义单元 |
| ② 漂移量化 | A101→LOC_DEMO_L101 子串 miss + 表层 0.33 不过阈；预喂别名后确定性命中 | 分界线第一段：能力面外靠治理，预喂别名（D3 生成规格的核心依据） |
| ③ 蒙特卡洛 | 治理修复零方差阶跃 / 校准修复留尾巴；剩余故障归因两分 | Today's Question：语义错→治理修，组装错→校准修 |

**给 D3 的直接输入**：生成术语条目时，编码风格值（A101 等口语铺号）必须进子别名——这是把「检索能力面外的映射」收编为「子串路确定性命中」的唯一途径；description 同时携带字段口径与 Rule 条件（L3 注入接口）。